# BT4012: Student-selected XGBoost feature-sampling experiment

**Date: 11 October 2026.** The student chose to test a three-model, 80%-feature XGBoost ensemble using four chronological checks after discussing the hypothesis and validation alternatives with OpenAI Codex.

The earlier review did not train a new candidate because the modeling decision was pending. It did not establish that the existing model was the best. This notebook now trains and evaluates the fixed experiment the student selected.

The best confirmed Kaggle public ROC-AUC remains **0.94265**, from the original tuned XGBoost submission. The time-filtered candidate and LightGBM have not been submitted. Local results below are development results, not Kaggle scores. Running these checks does not consume submissions.

This notebook reads only the supplied training data. It does not obtain external data, original labels, other participants' solutions, or hidden test answers. No automatic parameter search or final-model selection is performed.

## 1. Explain the experiment in simple terms

An XGBoost model combines many small decision trees. A tree asks questions about feature values; later trees adjust the predictions made by earlier trees.

**Feature sampling:** before building each tree, randomly give it 80% of the retained feature columns. The subset changes between trees. This can encourage alternative combinations of clues, although excluding useful clues can also hurt.

**Three models:** use fixed random seeds 42, 7, and 2026 to obtain different sampled subsets. A seed makes a randomized training run repeatable; it does not encode a fraud label. We evaluate every seed and do not choose whichever seed scores highest.

**Average probabilities:** if the three models predict 0.80, 0.65, and 0.75 for one transaction, their average is (0.80 + 0.65 + 0.75) / 3 = 0.7333. Averaging may reduce sensitivity to a particular model's choices. If all three make similar mistakes, it offers little help.

The hypothesis is motivated by the earlier diagnostic finding 21 feature pairs with absolute correlation above 0.995. Correlation is a reason to investigate redundant clues, not proof that sampling improves future performance.

The existing time-feature filter is kept: remove features whose absolute Spearman correlation with `time_step` exceeds 0.9 in the initial fitting period. Validation rows do not choose which features to remove. Feature names remain anonymized.

`colsample_bytree=0.8` implements per-tree feature sampling. [XGBoost parameter documentation](https://xgboost.readthedocs.io/en/release_3.2.0/parameter.html).

## 2. Why these four chronological checks?

Think of them as four practice exams. Each exam teaches the model using earlier transactions and asks it to predict the next fourteen periods.

| Check | Initial fitting and feature selection | Internal stopping | Refit before prediction | Outer validation |
|---|---|---|---|---|
| A | 1–11 | 12–14 | 1–14 | 15–28 |
| B | 1–14 | 15–17 | 1–17 | 18–31 |
| C | 1–16 | 17–19 | 1–19 | 20–33 |
| D | 1–18 | 19–21 | 1–21 | 22–35 |

The competition predicts steps 36–49, a fourteen-period horizon. These checks reproduce that horizon using labels we possess. We split by whole time steps because different steps contain different numbers of transactions.

The last three steps inside each training period choose the number of trees. We then train a fresh model on that whole training period for the chosen number of trees and score the later outer validation period. The internal stopping examples are included in this refit, while the outer answers remain excluded from fitting and feature selection.

| Validation option | What it can tell us | Why the selected checks suit this comparison |
|---|---|---|
| Random stratified splitting | Performance on withheld transactions mixed across the timeline | The actual task predicts later periods, so mixing periods answers a different question and can be optimistic |
| One chronological holdout | How one model performs on one future block | Four cutoffs show whether the result depends on one particular training endpoint |
| Ordinary k-fold splitting | Performance when each partition is withheld | Training partitions can include periods later than the validation partition |
| Recent-history sliding window | Whether discarding older training examples helps | This changes the training strategy as well as evaluation; earlier recent-window trials were not consistently better |
| Nested chronological evaluation | Separate settings/stopping selection from outer evaluation | We already use an inner chronological stopping period; a broad inner parameter search is unnecessary for this fixed comparison |

These cutoffs are useful rather than uniquely correct. Checks A and D have already influenced earlier model development. B and C overlap previously examined periods. The four scores are related development evidence, not four independent test datasets. Their spread is not a confidence interval. Any recommendation must acknowledge that the private Kaggle score is unknown.

Chronological validation is appropriate when the deployment task predicts future observations. [Scikit-learn validation guidance](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-of-time-series-data).

## 3. Freeze the student-selected settings

The original and time-filtered models provide two controls. Comparing the ensemble with the time-filtered control isolates feature sampling and averaging from the earlier feature-removal change.

We keep the original learning rate (0.05), maximum leaves per tree (15), bins (127), regularization (10), minimum child weight (3), row-sampling fraction (1.0), and positive-class weight (1.0). The maximum of 400 rounds and patience of 60 are the earlier chronological stopping settings. We use two CPU threads to keep the workload modest.

No setting, seed, ensemble weight, or cutoff is changed in response to these results.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import time

import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

# This fallback is for the student's current Mac. On another machine, put
# 'Data Set' beside the notebook or set DATA_DIR to the supplied files.
data_locations = [
    Path.cwd() / 'Data Set',
    Path.cwd().parent / 'Data Set',
    Path('/Users/mlvlo/BT4012/Kaggle/Data Set'),
]
DATA_DIR = next((p for p in data_locations if (p / 'Train Data.csv').is_file()), None)
assert DATA_DIR is not None, 'Set DATA_DIR to the supplied competition files.'
OUTPUT_DIR = Path.cwd() / 'feature_ensemble_results'
OUTPUT_DIR.mkdir(exist_ok=True)

FEATURES = [f'feat_{i}' for i in range(1, 166)]
CHECKS = [(14, 15, 28), (17, 18, 31), (19, 20, 33), (21, 22, 35)]
SEEDS = (42, 7, 2026)
FEATURE_FRACTION = 0.8
TIME_CORRELATION_LIMIT = 0.9
MAXIMUM_ROUNDS = 400
STOPPING_PATIENCE = 60
CPU_THREADS = 2

train_path = DATA_DIR / 'Train Data.csv'
train = pd.read_csv(train_path, usecols=['txId', 'time_step', 'label'] + FEATURES)
assert train.txId.is_unique
assert train.time_step.between(1, 35).all()
assert np.isfinite(train[FEATURES].to_numpy()).all()
known = train.loc[train.label.notna()].copy()
assert known.label.isin([0, 1]).all()
known['label'] = known.label.astype(int)
assert known.label.nunique() == 2
print('Known labels:', len(known), '; illicit:', int(known.label.sum()))
print('Source:', train_path)
print('XGBoost:', xgboost.__version__, '; CPU threads:', CPU_THREADS)


Known labels: 31235 ; illicit: 3644
Source: /Users/mlvlo/BT4012/Kaggle/Data Set/Train Data.csv
XGBoost: 3.2.0 ; CPU threads: 2


## 4. Model fitting and training-only feature filtering

Both controls and all three ensemble members use the same chronology. Each ensemble member selects its stopping round independently using only its internal stopping period. They share the same training-only retained feature list.

The outer validation labels are used only to calculate the reported scores. Identifier and time assertions protect against accidentally including the checking transactions in training.

In [2]:
def make_model(rounds, *, feature_fraction=1.0, seed=42, stopping=False):
    return XGBClassifier(
        n_estimators=int(rounds), learning_rate=0.05,
        tree_method='hist', device='cpu', grow_policy='lossguide',
        max_depth=0, max_leaves=15, max_bin=127,
        min_child_weight=3, reg_lambda=10,
        subsample=1.0, colsample_bytree=feature_fraction,
        scale_pos_weight=1.0, objective='binary:logistic', eval_metric='auc',
        early_stopping_rounds=STOPPING_PATIENCE if stopping else None,
        n_jobs=CPU_THREADS, random_state=seed,
    )


def select_features(learning):
    end = int(learning.time_step.max())
    fitting = learning.loc[learning.time_step <= end - 3]
    correlations = fitting[FEATURES].corrwith(fitting.time_step, method='spearman')
    removed = correlations.index[correlations.abs() > TIME_CORRELATION_LIMIT].tolist()
    retained = [f for f in FEATURES if f not in removed]
    assert retained and 'label' not in retained and 'time_step' not in retained
    return retained, removed


def fit_model(learning, columns, *, feature_fraction=1.0, seed=42):
    end = int(learning.time_step.max())
    fitting = learning.loc[learning.time_step <= end - 3]
    stopping = learning.loc[learning.time_step > end - 3]
    assert fitting.time_step.max() < stopping.time_step.min()
    assert fitting.label.nunique() == stopping.label.nunique() == 2
    with threadpool_limits(limits=CPU_THREADS):
        initial = make_model(MAXIMUM_ROUNDS, feature_fraction=feature_fraction,
                             seed=seed, stopping=True)
        initial.fit(fitting[columns], fitting.label,
                    eval_set=[(stopping[columns], stopping.label)], verbose=False)
        rounds = int(initial.best_iteration + 1)
        assert 1 <= rounds <= MAXIMUM_ROUNDS
        fitted = make_model(rounds, feature_fraction=feature_fraction, seed=seed)
        fitted.fit(learning[columns], learning.label)
    return fitted, rounds


## 5. Run precisely the selected comparison

For each cutoff, train the original control, time-filtered control, and three sampled-feature models. Save each model's outer validation probabilities. The three-model ensemble uses an equal-weight mean, fixed before looking at results.

The individual seed scores are diagnostic; the selected hypothesis concerns their average. The program does not automatically choose an individual seed or generate a final Kaggle submission.

In [3]:
records = []
saved_probabilities = {}
selection_records = {}
all_started = time.perf_counter()

for end, first, last in CHECKS:
    learning = known.loc[known.time_step <= end]
    checking = known.loc[known.time_step.between(first, last)]
    assert learning.time_step.max() < checking.time_step.min()
    assert set(learning.txId).isdisjoint(checking.txId)
    assert checking.label.nunique() == 2
    retained, removed = select_features(learning)
    selection_records[str(end)] = {
        'selection_steps': [1, end - 3], 'stopping_steps': [end - 2, end],
        'validation_steps': [first, last], 'retained_features': retained,
        'removed_features': removed,
    }
    saved_probabilities[f'labels_{end}'] = checking.label.to_numpy()
    saved_probabilities[f'time_steps_{end}'] = checking.time_step.to_numpy()
    saved_probabilities[f'txIds_{end}'] = checking.txId.to_numpy()
    members, member_rounds, member_seconds = [], [], []
    candidates = [
        ('original_xgboost', FEATURES, 1.0, 42),
        ('time_filtered_xgboost', retained, 1.0, 42),
    ] + [(f'sampled_seed_{seed}', retained, FEATURE_FRACTION, seed) for seed in SEEDS]

    for name, columns, fraction, seed in candidates:
        started = time.perf_counter()
        fitted, rounds = fit_model(learning, columns, feature_fraction=fraction, seed=seed)
        with threadpool_limits(limits=CPU_THREADS):
            probabilities = fitted.predict_proba(checking[columns])[:, 1]
        assert np.isfinite(probabilities).all()
        assert np.all((probabilities >= 0) & (probabilities <= 1))
        seconds = time.perf_counter() - started
        saved_probabilities[f'{name}_{end}'] = probabilities
        record = {
            'model': name, 'train_end': end, 'validation_start': first,
            'validation_end': last, 'learning_rows': len(learning),
            'checking_rows': len(checking), 'illicit_checking': int(checking.label.sum()),
            'feature_count': len(columns), 'feature_fraction': fraction,
            'rounds': str(rounds), 'auc': float(roc_auc_score(checking.label, probabilities)),
            'seconds': round(seconds, 3),
        }
        records.append(record)
        if name.startswith('sampled_seed_'):
            members.append(probabilities)
            member_rounds.append(rounds)
            member_seconds.append(seconds)
        print(f'{name}: learn 1–{end}, check {first}–{last}, '
              f'AUC={record["auc"]:.6f}, rounds={rounds}', flush=True)

    ensemble = np.mean(np.vstack(members), axis=0)
    saved_probabilities[f'feature_ensemble_{end}'] = ensemble
    records.append({
        'model': 'feature_ensemble', 'train_end': end, 'validation_start': first,
        'validation_end': last, 'learning_rows': len(learning),
        'checking_rows': len(checking), 'illicit_checking': int(checking.label.sum()),
        'feature_count': len(retained), 'feature_fraction': FEATURE_FRACTION,
        'rounds': ','.join(str(r) for r in member_rounds),
        'auc': float(roc_auc_score(checking.label, ensemble)),
        'seconds': round(sum(member_seconds), 3),
    })
    print(f'AVERAGE: learn 1–{end}, check {first}–{last}, '
          f'AUC={records[-1]["auc"]:.6f}; removed={removed}', flush=True)
    pd.DataFrame(records).to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
    np.savez_compressed(OUTPUT_DIR / 'validation_probabilities.npz', **saved_probabilities)
    (OUTPUT_DIR / 'feature_selections.json').write_text(json.dumps(selection_records, indent=2))

results = pd.DataFrame(records)
print('Total comparison seconds:', round(time.perf_counter() - all_started, 2))


original_xgboost: learn 1–14, check 15–28, AUC=0.968521, rounds=185
time_filtered_xgboost: learn 1–14, check 15–28, AUC=0.967511, rounds=186
sampled_seed_42: learn 1–14, check 15–28, AUC=0.967958, rounds=184
sampled_seed_7: learn 1–14, check 15–28, AUC=0.966671, rounds=160
sampled_seed_2026: learn 1–14, check 15–28, AUC=0.966723, rounds=146
AVERAGE: learn 1–14, check 15–28, AUC=0.967231; removed=['feat_101', 'feat_103', 'feat_136', 'feat_139']
original_xgboost: learn 1–17, check 18–31, AUC=0.925611, rounds=336
time_filtered_xgboost: learn 1–17, check 18–31, AUC=0.922779, rounds=349
sampled_seed_42: learn 1–17, check 18–31, AUC=0.922773, rounds=342
sampled_seed_7: learn 1–17, check 18–31, AUC=0.928180, rounds=337
sampled_seed_2026: learn 1–17, check 18–31, AUC=0.924291, rounds=328
AVERAGE: learn 1–17, check 18–31, AUC=0.925527; removed=['feat_100', 'feat_101', 'feat_103', 'feat_136', 'feat_139']
original_xgboost: learn 1–19, check 20–33, AUC=0.901582, rounds=398
time_filtered_xgboost: l

## 6. Compare the ensemble with both controls

Each AUC uses all known labels in its validation block. The mean gives equal weight to the four chronological checks; the minimum is the worst full-block score. Neither number is an independent estimate of the Kaggle score. We do not pool duplicated transactions from overlapping checks into a single AUC.

Positive differences mean an improvement; negative differences mean a decline. For the experiment's hypothesis, the time-filtered control is especially relevant because both it and the ensemble use the same filtered features.

In [4]:
primary_models = ['original_xgboost', 'time_filtered_xgboost', 'feature_ensemble']
comparison = results.pivot(index='train_end', columns='model', values='auc')[primary_models]
comparison['ensemble_minus_original'] = comparison.feature_ensemble - comparison.original_xgboost
comparison['ensemble_minus_time_filtered'] = comparison.feature_ensemble - comparison.time_filtered_xgboost
print(comparison.round(6).to_string())
display(comparison.round(6))

summary = comparison[primary_models].agg(['mean', 'min']).T
summary['checks_above_0.96'] = (comparison[primary_models] > 0.96).sum()
print('\nAcross the four development checks:')
print(summary.round(6).to_string())
display(summary.round(6))
comparison.to_csv(OUTPUT_DIR / 'comparison.csv')
summary.to_csv(OUTPUT_DIR / 'summary.csv')

# Agreement of the seeds helps explain whether averaging adds much diversity.
agreement_records = []
for end, first, last in CHECKS:
    matrix = np.vstack([saved_probabilities[f'sampled_seed_{seed}_{end}'] for seed in SEEDS])
    correlation = np.corrcoef(matrix)
    agreement_records.append({
        'train_end': end,
        'mean_pairwise_probability_correlation': float(correlation[np.triu_indices(3, 1)].mean()),
        'mean_across_seed_probability_sd': float(matrix.std(axis=0).mean()),
    })
agreement = pd.DataFrame(agreement_records)
print('\nHow similar were the three models?')
print(agreement.round(6).to_string(index=False))
agreement.to_csv(OUTPUT_DIR / 'seed_agreement.csv', index=False)


model      original_xgboost  time_filtered_xgboost  feature_ensemble  ensemble_minus_original  ensemble_minus_time_filtered
train_end                                                                                                                  
14                 0.968521               0.967511          0.967231                -0.001290                     -0.000279
17                 0.925611               0.922779          0.925527                -0.000085                      0.002747
19                 0.901582               0.894558          0.904005                 0.002423                      0.009448
21                 0.915330               0.921428          0.914804                -0.000526                     -0.006625

Across the four development checks:
                           mean       min  checks_above_0.96
model                                                       
original_xgboost       0.927761  0.901582                  1
time_filtered_xgboost  0.926569  0.8

model,original_xgboost,time_filtered_xgboost,feature_ensemble,ensemble_minus_original,ensemble_minus_time_filtered
train_end,,,,,
14,0.968521,0.967511,0.967231,-0.001290,-0.000279
17,0.925611,0.922779,0.925527,-0.000085,0.002747
19,0.901582,0.894558,0.904005,0.002423,0.009448
21,0.915330,0.921428,0.914804,-0.000526,-0.006625


,mean,min,checks_above_0.96
model,,,
original_xgboost,0.927761,0.901582,1
time_filtered_xgboost,0.926569,0.894558,1
feature_ensemble,0.927892,0.904005,1


## 7. Look inside the difficult periods

The main model comparison uses the pooled AUC of each fourteen-step block, matching the competition's probability-ranking objective. A score for each individual time step helps explain where mistakes occur. A step containing just one label class cannot have a defined AUC and is recorded as missing.

In [5]:
step_records = []
for end, first, last in CHECKS:
    labels = saved_probabilities[f'labels_{end}']
    steps = saved_probabilities[f'time_steps_{end}']
    for step in np.unique(steps):
        mask = steps == step
        row = {'train_end': end, 'validation_step': int(step),
               'known_labels': int(mask.sum()), 'illicit': int(labels[mask].sum())}
        for name in primary_models:
            p = saved_probabilities[f'{name}_{end}'][mask]
            row[name] = float(roc_auc_score(labels[mask], p)) if len(np.unique(labels[mask])) == 2 else np.nan
        row['ensemble_minus_time_filtered'] = row['feature_ensemble'] - row['time_filtered_xgboost']
        step_records.append(row)
per_step = pd.DataFrame(step_records)
per_step.to_csv(OUTPUT_DIR / 'per_time_step.csv', index=False)
print(per_step.loc[per_step.train_end == 21].round(6).to_string(index=False))
display(per_step.loc[per_step.train_end == 21].round(6))


 train_end  validation_step  known_labels  illicit  original_xgboost  time_filtered_xgboost  feature_ensemble  ensemble_minus_time_filtered
        21               22          1763      158          0.998833               0.998900          0.998975                      0.000075
        21               23          1187       53          0.957622               0.958570          0.958870                      0.000299
        21               24          1126      137          0.795488               0.820980          0.807806                     -0.013174
        21               25           594      118          0.999484               0.999573          0.999573                      0.000000
        21               26           517       96          0.946779               0.951653          0.947521                     -0.004132
        21               27           206       24          0.999542               0.999542          0.999542                      0.000000
        21          

,train_end,validation_step,known_labels,illicit,original_xgboost,time_filtered_xgboost,feature_ensemble,ensemble_minus_time_filtered
42,21,22,1763,158,0.998833,0.998900,0.998975,0.000075
43,21,23,1187,53,0.957622,0.958570,0.958870,0.000299
44,21,24,1126,137,0.795488,0.820980,0.807806,-0.013174
45,21,25,594,118,0.999484,0.999573,0.999573,0.000000
46,21,26,517,96,0.946779,0.951653,0.947521,-0.004132
47,21,27,206,24,0.999542,0.999542,0.999542,0.000000
48,21,28,284,85,1.000000,1.000000,1.000000,0.000000
49,21,29,1174,329,0.775619,0.820500,0.801263,-0.019237
50,21,30,524,83,0.987323,0.987979,0.988853,0.000874
51,21,31,710,106,0.772820,0.770102,0.768228,-0.001874


## 8. Verify and record the experiment

Save the settings, data-file hash, all probabilities, and results so the comparison can be explained and reproduced. These checks verify chronology, probabilities, averaging, and the displayed metrics. They do not reveal hidden Kaggle answers.

In [6]:
for end, first, last in CHECKS:
    labels = saved_probabilities[f'labels_{end}']
    txids = saved_probabilities[f'txIds_{end}']
    expected_ids = known.loc[known.time_step.between(first, last), 'txId'].to_numpy()
    assert np.array_equal(txids, expected_ids)
    expected_average = np.mean(np.vstack([
        saved_probabilities[f'sampled_seed_{seed}_{end}'] for seed in SEEDS
    ]), axis=0)
    assert np.array_equal(expected_average, saved_probabilities[f'feature_ensemble_{end}'])
    for name in primary_models:
        recalculated = roc_auc_score(labels, saved_probabilities[f'{name}_{end}'])
        assert abs(recalculated - comparison.loc[end, name]) < 1e-12

hasher = hashlib.sha256()
with train_path.open('rb') as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b''):
        hasher.update(chunk)
metadata = {
    'date': '2026-10-11', 'experiment_decision': 'student chose fixed three-model 80% feature ensemble',
    'validation_decision': 'student chose four chronological checks',
    'checks': CHECKS, 'seeds': SEEDS, 'feature_fraction': FEATURE_FRACTION,
    'feature_filter': 'abs(Spearman correlation with time_step) > 0.9 on initial fitting rows only',
    'settings': {key: (None if isinstance(value, float) and not np.isfinite(value) else value)
                 for key, value in make_model(400).get_params().items()
                 if value is None or isinstance(value, (str, int, float, bool))},
    'maximum_rounds': MAXIMUM_ROUNDS, 'stopping_patience': STOPPING_PATIENCE,
    'known_labels': len(known), 'training_csv_sha256': hasher.hexdigest(),
    'python': platform.python_version(), 'xgboost': xgboost.__version__,
    'sklearn': sklearn.__version__, 'pandas': pd.__version__, 'numpy': np.__version__,
    'outer_checks_overlap': True, 'outer_checks_are_development_evidence': True,
    'previous_best_public_auc': 0.94265, 'candidate_public_auc': None,
    'final_model_decision': 'pending student review', 'kaggle_submission_made': False,
}
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2, allow_nan=False))
print('Chronology, transaction alignment, probability averaging, and AUC checks passed.')
print('Saved results:', OUTPUT_DIR)


Chronology, transaction alignment, probability averaging, and AUC checks passed.
Saved results: /Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/feature_ensemble_results


## 9. Measured result and recommendation

All 20 individual models (two controls plus three members at each of four cutoffs) were trained. The three sampled members were averaged at each cutoff. All notebook code cells ran in sequence in a fresh Python process. Transaction alignment, averaging, AUC recalculation, and reproduction of the historical controls passed. Local testing used two CPU threads; no Kaggle submission was made.

| Validation steps | Original XGBoost | Time-filtered XGBoost | Three-model ensemble |
|---|---:|---:|---:|
| 15–28 | 0.968521 | 0.967511 | 0.967231 |
| 18–31 | 0.925611 | 0.922779 | 0.925527 |
| 20–33 | 0.901582 | 0.894558 | 0.904005 |
| 22–35 | 0.915330 | 0.921428 | 0.914804 |
| **Mean of four checks** | **0.927761** | **0.926569** | **0.927892** |

**The experiment did not demonstrate a consistent advantage over the original model.** The ensemble improved 1 of four checks and declined on 3. Its mean advantage was only 0.000130 AUC. Its worst full-block result improved from 0.901582 to 0.904005, but it declined on the check ending at step 35. We have not established whether the small mean difference would carry into the actual future test period.

The time-feature filter alone improved 1 of four checks and declined on 3. On the two older checks its mean looked better, but the additional cutoffs weakened that evidence. This is a concrete reason to examine several time cutoffs before choosing a submission. It does not prove that time filtering will hurt the unknown Kaggle test set.

The mean pairwise correlation of the three members' validation probabilities was 0.9987. Their similar predictions limit the diversity that averaging can contribute. This is descriptive evidence, not a proof of the cause of the outcome.

**AI-assisted recommendation:** do not replace the original tuned XGBoost solely on the basis of this experiment. Do not select one seed because its individual score happened to be higher. The student should review the results before choosing a final model or allocating a remaining submission. A fresh hypothesis, if chosen, needs its own student decision rather than an automatic expansion of this experiment.

The original model remains the best *confirmed public* result at 0.94265. We have not demonstrated 0.96+ on Kaggle or across all these development checks. The leading score reported by the student is 0.968; it is a target, not evidence that this pipeline can match it.

### Decision record

- **Student-selected experiment:** three XGBoost models, each sampling 80% of filtered features per tree, with fixed seeds 42, 7, and 2026 and equal averaging.
- **Student-selected validation:** the four fourteen-step chronological checks above, with an internal three-step stopping period.
- **Fixed scope:** earlier XGBoost settings, no parameter or seed search, no external data, no Kaggle upload.
- **Observed result:** inconsistent improvement; the mean difference is small and the checks overlap.
- **Final-model and next-submission decision:** pending the student's review.

### AI assistance and ownership

OpenAI Codex helped inspect the existing results, calculate a correlation diagnostic, explain feature sampling and validation alternatives, and implement and run this fixed comparison. The student explicitly selected the experiment and validation strategy after the explanation. Codex summarized the measured results and prepared this journal addition. The student must review and explain the interpretation and make the final model/submission decision. The report should disclose this assistance accurately.

### Questions to answer in the student's own words

1. Why did I choose to test feature sampling and averaging?
2. What does 80% sampling change for a tree, and why might it help or hurt?
3. Why did I use fixed seeds and average every member instead of choosing the highest-scoring seed?
4. Why do later periods provide a more relevant check than randomly mixed transactions?
5. Why are overlapping, repeatedly examined checks limited evidence?
6. Why did I keep or reject the ensemble after reviewing these results?
